In [1]:
%pip install -q langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
%autoreload 2
    
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ('LANGCHAIN_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = getpass.getpass("Enter LangSmith API Key: ")
    
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'rag-lab-query-translation'

In [8]:
# Loading the Vectorstore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma   
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini")
)
    
print("Chunks in store:", vectorstore._collection.count())

#llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

Chunks in store: 979


In [4]:
# Prompting the LLM for the hypothetical document
from langchain_core.prompts import ChatPromptTemplate

HYDE_PROMPT = ChatPromptTemplate.from_template(
    "Please write a passage that could plausibly appear in a technical "
    "paper, answering the question below. Write only the passage itself "
    "— no preamble, no meta-commentary, no acknowledgment that this is "
    "hypothetical.\n\n"
    "Question: {question}\n\n"
    "Passage:"
)

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.language_models import BaseChatModel
from langchain_core.vectorstores import VectorStore
from typing import List


class HydeStrategy:
    """Generates a hypothetical answer document, embeds THAT instead
    of the raw query, and retrieves by similarity to the hypothetical
    answer rather than to the question itself."""

    def __init__(self, vectorstore: VectorStore, llm: BaseChatModel, k: int = 4):
        self.vectorstore = vectorstore
        self.llm = llm
        self.k = k
        self.hyde_chain = HYDE_PROMPT | llm | StrOutputParser()
        self.answer_prompt = ChatPromptTemplate.from_template(
            "Answer the question based only on the following context:\n"
            "{context}\n\nQuestion: {question}"
        )

    def generate_hypothetical_doc(self, query: str) -> str:
        return self.hyde_chain.invoke({"question": query})

    def retrieve(self, query: str) -> List[Document]:
        hypothetical_doc = self.generate_hypothetical_doc(query)
        # similarity_search embeds `hypothetical_doc` and searches against it —
        # this is the key structural difference from the previous strategies,
        # which called .invoke(query) on a pre-built retriever using the raw query
        return self.vectorstore.similarity_search(hypothetical_doc, k=self.k)

    def run(self, query: str) -> str:
        docs = self.retrieve(query)
        context = "\n\n".join(d.page_content for d in docs)
        chain = self.answer_prompt | self.llm | StrOutputParser()
        return chain.invoke({"context": context, "question": query})

In [7]:
from rag_lab.strategies.hyde import HydeStrategy
from rag_lab.utils import normalize_text

test_query = "Why do PINNs struggle with irregular geometry?"
strategy = HydeStrategy(vectorstore, llm=llm)

hypothetical_doc = strategy.generate_hypothetical_doc(test_query)

#non_ascii = [(i, c, hex(ord(c))) for i, c in enumerate(hypothetical_doc) if ord(c) > 127]
#print(f"Found {len(non_ascii)} non-ASCII characters:")
#for i, c, code in non_ascii[:20]:  # first 20 is plenty to see the pattern
#    print(f"  position {i}: {repr(c)} ({code})")

cleaned = normalize_text(hypothetical_doc)

print("Generated hypothetical document:\n")
print(hypothetical_doc)
print("\n" + "="*60 + "\n")

docs = strategy.retrieve(test_query)
print(f"Retrieved {len(docs)} docs based on similarity to the hypothetical doc:\n")
for i, doc in enumerate(docs):
    print(f"{i+1}. {doc.page_content[:150]}...\n")

Generated hypothetical document:

The difficulty of applying physics-informed neural networks (PINNs) to domains with irregular geometry stems from several intertwined factors that affect both the representation of the computational domain and the enforcement of boundary conditions. First, PINNs rely on a set of collocation points drawn from a reference coordinate space; when the physical domain possesses complex, non-convex, or highly curved boundaries, mapping these points uniformly onto the irregular region becomes non-trivial. Standard sampling strategies (e.g., uniform random or Latin hypercube sampling) either oversample interior regions or leave large portions of the boundary under-represented, leading to poor approximation of the governing PDE near geometric features such as re-entrant corners or thin channels. 

Second, the neural network must implicitly learn a characteristic function that distinguishes interior from exterior points. In the absence of an explicit mesh, the ne